### 1. 使用dataprocessor对数据进行预处理操作

In [1]:
# %% 初始化环境
import sys
sys.path.append("..")
from experiments.environment import *
init_environment()
from pathlib import Path

Applied font.family: ['Times New Roman', 'Microsoft YaHei', 'DejaVu Sans']
findfont Times -> C:\Windows\Fonts\times.ttf
findfont CN -> C:\Windows\Fonts\msyh.ttc
🚀 环境初始化完成。


In [2]:
dataset_name = ['breast_cancer', 'diabetes_risk', 'framingham', 'hepatitis',
                'pima','tcga', 'thyroid']

train_ratio, val_ratio, test_ratio = 0.7, 0.1, 0.2
assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6 , "训练、验证和测试集比例之和必须为1"

In [ ]:
import json
from collections import Counter
from sklearn.model_selection import train_test_split
from core.data_processor import TabularDataProcessor


for i in range(5):
    loc = Path('../splitted_data')
    splitted_loc = loc/f"seed_{i}"
    dataset_info_loc = Path('./data/dataset_info.json')

    with open(dataset_info_loc, 'r', encoding='utf-8') as f:
        info = json.load(f)
    
    for name in dataset_name:
        # 1. 读取数据集
        loc = './data/'+ name + '.csv'
        df_all = pd.read_csv(loc)

        # 2. 读取属性特性
        cat_cols = info[name]["categorical"]
        num_cols = info[name]["numerical"]
        target_col = info[name]["target"]

        # 2.5 定义预处理器
        processor = TabularDataProcessor(num_cols=num_cols, cat_cols=cat_cols, 
                                         num_transform_method='quantile', cat_encoding_method='ordinal',
                                         random_state=i + 1)
        
        # 3. 划分数据集
        unique_classes = df_all[target_col].nunique()
        stratify_col = df_all[target_col] if unique_classes > 1 else None

        df_temp, df_test = train_test_split(
                df_all,
                test_size=test_ratio,
                stratify=stratify_col,
                random_state=i,
            )
        
        stratify_col_temp = df_temp[target_col] if unique_classes > 1 else None 
        val_ratio_in_temp = val_ratio / (train_ratio + val_ratio)

        df_train, df_val = train_test_split(
            df_temp,
            test_size=val_ratio_in_temp,
            stratify=stratify_col_temp,
            random_state=i + 42
        )

        # 4. 缺失值补全
        processor.fit(df_train) # 使用训练集拟合
        real_train_imputed = processor.transform_readable(df_train)
        real_val_imputed = processor.transform_readable(df_val)
        real_test_imputed = processor.transform_readable(df_test)

        loc = splitted_loc / name
        loc.mkdir(parents=True, exist_ok=True)

        real_train_imputed.to_csv( loc/ "real_train_imputed.csv", index=False)
        real_val_imputed.to_csv(loc / "real_valid_imputed.csv", index=False)
        real_test_imputed.to_csv(loc / "real_test_imputed.csv", index=False)        


pd.read_csv("../splitted_data/seed_0/pima/real_valid_imputed.csv")

In [5]:
for name in dataset_name:
    loc = Path(f"../splitted_data/seed_0/{name}")

    train = pd.read_csv(loc / "real_train_imputed.csv")
    valid = pd.read_csv(loc / "real_valid_imputed.csv")
    test = pd.read_csv(loc / "real_test_imputed.csv")

    save_loc = Path(f"../data/{name}")
    save_loc.mkdir(parents=True, exist_ok=True)

    pd.concat([train, valid, test], ignore_index=True).to_csv(save_loc / f"{name}.csv", index=False)

In [6]:
import pandas as pd
import json
from pathlib import Path

# 1. 读取配置
with open("./data/dataset_info.json", "r", encoding="utf-8") as f:
    datasets_cfg = json.load(f)

# 2. 假设每个数据集有对应的 CSV 文件（路径按你的实际情况调整）
data_dir = Path("./data")  # 存放原始 CSV 的目录

idx_mapping = {}

for name, info in datasets_cfg.items():
    # 读取真实数据（只读取列名，不加载全部数据以节省内存）
    csv_path = data_dir / f"{name}.csv"       # 根据实际文件名调整
    df_cols = pd.read_csv(csv_path, nrows=0).columns.tolist()

    # 获取各类型列的索引
    cat_cols   = info["categorical"]
    num_cols   = info["numerical"]
    target_col = info["target"]

    cat_idx   = [df_cols.index(c) for c in cat_cols]
    num_idx   = [df_cols.index(c) for c in num_cols]
    target_idx = [df_cols.index(target_col)]

    idx_mapping[name] = {
        "cat_col_idx": cat_idx,
        "num_col_idx": num_idx,
        "target_col_idx": target_idx
    }

# 3. 保存为新 JSON
output_path = Path("./data/datasets_idx.json")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(idx_mapping, f, indent=4)

print("已生成索引映射文件:", output_path)

已生成索引映射文件: data\datasets_idx.json
